<a href="https://colab.research.google.com/github/Pratyakshk05/deep-learning/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import itertools
import os


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                             download=True, transform=transform)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                            download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

num_classes = 10


100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


In [ ]:
class CNNModel(nn.Module):
    def __init__(self, activation_fn):
        super(CNNModel, self).__init__()

        self.activation = activation_fn

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            self.activation,
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            self.activation,
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            self.activation,
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*8*8, 256),
            self.activation,
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:
def initialize_weights(model, init_type):
    for m in model.modules():
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
            if init_type == "xavier":
                nn.init.xavier_uniform_(m.weight)
            elif init_type == "kaiming":
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            elif init_type == "random":
                nn.init.normal_(m.weight, 0, 0.02)


In [ ]:
import os

os.makedirs("saved_models", exist_ok=True)


In [ ]:
def train_model(model, optimizer, criterion, epochs=10):
    model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")


In [ ]:
def evaluate_model(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy


In [ ]:
activations = {
    "relu": nn.ReLU(),
    "tanh": nn.Tanh(),
    "leakyrelu": nn.LeakyReLU()
}

initializations = ["xavier", "kaiming", "random"]
optimizers_list = ["sgd", "adam", "rmsprop"]

criterion = nn.CrossEntropyLoss()

best_acc = 0
best_model = None

for act_name, act_fn in activations.items():
    for init in initializations:
        for opt_name in optimizers_list:

            print(f"\nRunning: {act_name}, {init}, {opt_name}")

            model = CNNModel(act_fn)
            initialize_weights(model, init)

            if opt_name == "sgd":
                optimizer = optim.SGD(model.parameters(), lr=0.01)
            elif opt_name == "adam":
                optimizer = optim.Adam(model.parameters(), lr=0.001)
            elif opt_name == "rmsprop":
                optimizer = optim.RMSprop(model.parameters(), lr=0.001)

            train_model(model, optimizer, criterion)
            acc = evaluate_model(model)

            if acc > best_acc:
                best_acc = acc
                best_model = model
                torch.save(model.state_dict(), "saved_models/best_cnn.pth")


print("Best Accuracy:", best_acc)



Running: relu, xavier, sgd
Epoch [1/10], Loss: 1.6332
Epoch [2/10], Loss: 1.3434
Epoch [3/10], Loss: 1.2076
Epoch [4/10], Loss: 1.1010
Epoch [5/10], Loss: 1.0176
Epoch [6/10], Loss: 0.9530
Epoch [7/10], Loss: 0.9052
Epoch [8/10], Loss: 0.8560
Epoch [9/10], Loss: 0.8128
Epoch [10/10], Loss: 0.7781
Test Accuracy: 69.83%

Running: relu, xavier, adam
Epoch [1/10], Loss: 1.9547
Epoch [2/10], Loss: 1.6648
Epoch [3/10], Loss: 1.5638
Epoch [4/10], Loss: 1.4992
Epoch [5/10], Loss: 1.4607
Epoch [6/10], Loss: 1.4247
Epoch [7/10], Loss: 1.3806
Epoch [8/10], Loss: 1.3629
Epoch [9/10], Loss: 1.3359
Epoch [10/10], Loss: 1.3064
Test Accuracy: 67.25%

Running: relu, xavier, rmsprop
Epoch [1/10], Loss: 2.3688
Epoch [2/10], Loss: 1.6812
Epoch [3/10], Loss: 1.5254
Epoch [4/10], Loss: 1.4319
Epoch [5/10], Loss: 1.3589
Epoch [6/10], Loss: 1.2872
Epoch [7/10], Loss: 1.2283
Epoch [8/10], Loss: 1.1709
Epoch [9/10], Loss: 1.1285
Epoch [10/10], Loss: 1.0863
Test Accuracy: 67.05%

Running: relu, kaiming, sgd
Epo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision.models as models

# ===== DEVICE =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== DATA (CIFAR-10) =====
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),   # Important for ResNet
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ===== LOAD RESNET =====
resnet = models.resnet18(pretrained=True)

# Unfreeze all layers
for param in resnet.parameters():
    param.requires_grad = True

# Replace final layer
resnet.fc = nn.Linear(resnet.fc.in_features, 10)

resnet = resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=0.0003)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# ===== TRAIN =====
def train_resnet(model, epochs=20):
    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        scheduler.step()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# ===== EVALUATE =====
def evaluate_resnet(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

# ===== RUN =====
train_resnet(resnet, epochs=20)
acc = evaluate_resnet(resnet)

print("Final ResNet Accuracy:", acc)


100%|██████████| 170M/170M [00:03<00:00, 43.7MB/s]


Epoch [1/20], Loss: 0.3568
Epoch [2/20], Loss: 0.1954
Epoch [3/20], Loss: 0.1490
Epoch [4/20], Loss: 0.1157
Epoch [5/20], Loss: 0.0950
Epoch [6/20], Loss: 0.0779
Epoch [7/20], Loss: 0.0719
Epoch [8/20], Loss: 0.0301
Epoch [9/20], Loss: 0.0137
Epoch [10/20], Loss: 0.0081
Epoch [11/20], Loss: 0.0059
Epoch [12/20], Loss: 0.0034
Epoch [13/20], Loss: 0.0028
Epoch [14/20], Loss: 0.0021
Epoch [15/20], Loss: 0.0017
Epoch [16/20], Loss: 0.0015
Epoch [17/20], Loss: 0.0011
Epoch [18/20], Loss: 0.0010
Epoch [19/20], Loss: 0.0011
Epoch [20/20], Loss: 0.0010
Test Accuracy: 96.02%
Final ResNet Accuracy: 96.02
